# 板块三：聚合运算（Reduction Operations）

## 一、基础聚合函数  
所有聚合函数都支持 dim 参数，指定沿哪个维度进行聚合。聚合后该维度消失（或可保留）

| 函数                              | 作用                             |
|-----------------------------------|----------------------------------|
| `torch.sum()`                     | 求和                             |
| `torch.mean()`                    | 求平均                           |
| `torch.max()` / `torch.min()`     | 最大值/最小值（返回值和索引）    |
| `torch.argmax()` / `torch.argmin()` | 返回最大/最小值的索引           |

## 二、代码详解 + 示例
###  1. 不指定维度 → 全局聚合

In [1]:
import torch

x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])  # shape: (2, 3)

print("Total sum:", x.sum())        # tensor(21.)
print("Total mean:", x.mean())      # tensor(3.5)

Total sum: tensor(21.)
Total mean: tensor(3.5000)


### 2. 指定维度 dim=0 → 沿行方向聚合（压缩行，保留列）

In [3]:
print("Sum over dim=0:", x.sum(dim=0))    # [5., 7., 9.]
# 解释：第0列：1+4=5，第1列：2+5=7，第2列：3+6=9

print("Mean over dim=0:", x.mean(dim=0))  # [2.5, 3.5, 4.5]

Sum over dim=0: tensor([5., 7., 9.])
Mean over dim=0: tensor([2.5000, 3.5000, 4.5000])


### 3. 指定维度 dim=1 → 沿列方向聚合（压缩列，保留行）

In [4]:
print("Sum over dim=1:", x.sum(dim=1))    # [6., 15.]
# 第0行：1+2+3=6，第1行：4+5+6=15

print("Mean over dim=1:", x.mean(dim=1))  # [2., 5.]

Sum over dim=1: tensor([ 6., 15.])
Mean over dim=1: tensor([2., 5.])


### 4. 保留维度（keepdim=True）

In [5]:
sum_dim1 = x.sum(dim=1, keepdim=True)  # shape: (2, 1)
print("With keepdim:\n", sum_dim1)
# 输出：
# tensor([[ 6.],
#         [15.]])

# 这样就可以与原张量做除法（如行归一化）
normalized = x / sum_dim1
print("Row-normalized:\n", normalized)
# 每行和为1

With keepdim:
 tensor([[ 6.],
        [15.]])
Row-normalized:
 tensor([[0.1667, 0.3333, 0.5000],
        [0.2667, 0.3333, 0.4000]])


## 三、max 和 argmax 的特殊用法  
### torch.max() 返回 值和索引，而 torch.argmax() 只返回索引。

In [6]:
x = torch.tensor([[1., 4., 3.],
                  [2., 1., 5.]])

# 获取每行最大值及其位置
values, indices = x.max(dim=1)
print("Max values:", values)   # [4., 5.]
print("Argmax indices:", indices)  # [1, 2]

# 只要索引（常用于分类预测）
pred_class = x.argmax(dim=1)
print("Predicted classes:", pred_class)  # tensor([1, 2])

Max values: tensor([4., 5.])
Argmax indices: tensor([1, 2])
Predicted classes: tensor([1, 2])


## 四、实际应用场景

### 场景1：计算分类准确率

In [7]:
# 模拟 CNN 特征图: (batch, channels, H, W)
feat = torch.randn(4, 64, 8, 8)

# 对每个通道的空间维度(H,W)求均值 → (4, 64)
gap = feat.mean(dim=[2, 3])  # 沿 dim=2 和 dim=3 聚合
print("After GAP:", gap.shape)  # torch.Size([4, 64])

After GAP: torch.Size([4, 64])


### 场景2：全局平均池化（Global Average Pooling）

In [8]:
# 模拟 CNN 特征图: (batch, channels, H, W)
feat = torch.randn(4, 64, 8, 8)

# 对每个通道的空间维度(H,W)求均值 → (4, 64)
gap = feat.mean(dim=[2, 3])  # 沿 dim=2 和 dim=3 聚合
print("After GAP:", gap.shape)  # torch.Size([4, 64])

After GAP: torch.Size([4, 64])


### 场景3：计算 MSE 损失（手动实现）

In [9]:
pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.5, 2.5, 3.5])

mse_loss = (pred - target).pow(2).mean()  # 先平方，再求均值
print("MSE loss:", mse_loss.item())       # 0.25

MSE loss: 0.25


## 五、动手练习

In [10]:
# 练习1：计算每列的最大值及其所在行索引
x = torch.tensor([[10, 20, 30],
                  [40, 15, 25]])

col_max_vals, col_max_idx = x.max(dim=0)
print("Column max values:", col_max_vals)    # [40, 20, 30]
print("Row indices of max:", col_max_idx)    # [1, 0, 0]

# 练习2：实现 softmax（不使用 nn.Softmax）
logits = torch.tensor([2.0, 1.0, 0.1])

# 步骤：exp(x) / sum(exp(x))
exp_x = torch.exp(logits - logits.max())  # 减最大值防溢出（数值稳定）
softmax_manual = exp_x / exp_x.sum()
print("Manual softmax:", softmax_manual)

# 验证
from torch.nn import Softmax
softmax_layer = Softmax(dim=0)
print("nn.Softmax:", softmax_layer(logits))

Column max values: tensor([40, 20, 30])
Row indices of max: tensor([1, 0, 0])
Manual softmax: tensor([0.6590, 0.2424, 0.0986])
nn.Softmax: tensor([0.6590, 0.2424, 0.0986])


# 总结

| 要点             | 说明                                               |
|------------------|----------------------------------------------------|
| `dim` 含义       | 沿该维度“压缩”，结果不再包含此维度                 |
| `keepdim=True`   | 保留被压缩的维度（大小变为1），便于广播            |
| `max` vs `argmax`| 前者返回值+索引，后者只返回索引                    |
| 多维度聚合       | `dim=[2,3]` 可同时沿多个维度聚合                   |
| 数值稳定性       | 如 softmax 中减去最大值                            |